# Choosing Visual Forms — class notebook

**36104 Data Visualisation and Narratives · Class 2**

Good chart choice begins with the analytical task, not with the visual effect.
In this notebook you will feel the difference between encodings, reshape data
into the form charts want, classify tasks with the FT Visual Vocabulary, and
redesign a weak chart three ways.

## How to work in this notebook

Same contract as Class 1: AI assistant on, docstrings first, and **nothing
counts until its verification cell passes**. The five wrangling checks from
the lecture — identifiers, row count, types, missing values, spot totals —
appear here as executable assertions.

In [ ]:
# Setup — run this cell, no need to edit it.
# It builds the synthetic public-transport patronage dataset used across this course:
# monthly rider counts for six NSW regions and four transport modes, 2019-2025,
# with a seasonal cycle and a COVID-shaped shock. Teaching data, not real data.
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

REGIONS = ["Inner Sydney", "Western Sydney", "Northern Beaches",
           "Central Coast", "Newcastle", "Illawarra"]
MODES = ["Train", "Bus", "Ferry", "Light rail"]
BASE = {"Train": 1_000_000, "Bus": 700_000, "Ferry": 90_000, "Light rail": 120_000}
FACTOR = {"Inner Sydney": 1.3, "Western Sydney": 1.1, "Northern Beaches": 0.55,
          "Central Coast": 0.45, "Newcastle": 0.5, "Illawarra": 0.42}

rows = []
for region in REGIONS:
    for mode in MODES:
        base = BASE[mode] * FACTOR[region]
        for date in pd.date_range("2019-01-01", "2025-12-01", freq="MS"):
            season = 1 + 0.08 * math.sin((date.month - 1) / 12 * 2 * math.pi)
            covid = 1.0
            if pd.Timestamp("2020-03-01") <= date <= pd.Timestamp("2021-12-01"):
                covid = 0.35 + 0.3 * (date - pd.Timestamp("2020-03-01")).days / 640
            elif date > pd.Timestamp("2021-12-01"):
                covid = min(1.0, 0.65 + 0.35 * (date - pd.Timestamp("2021-12-01")).days / 1100)
            noise = random.gauss(1, 0.03)
            rows.append({"date": date, "region": region, "mode": mode,
                         "riders": int(base * season * covid * noise)})

transport = pd.DataFrame(rows)
print(f"{len(transport):,} rows")
transport.head()

## Exercise 1 — Channels are read at different precision

The same five values, encoded five ways. Run the cell, then — *before any
computation* — estimate the ratio of value B to value D from each panel alone,
and record your five estimates.

Cleveland & McGill's ranking predicts your position estimate will be sharpest
and your area/colour estimates worst. Let's test that on you.

In [ ]:
# Run, then look — do not read the values from the code before estimating!
values = np.array([34, 87, 51, 29, 66])
labels = list("ABCDE")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
axes[0].scatter(values, labels, s=60, color="#18678f")          # position
axes[0].set_title("position"); axes[0].set_xlim(0, 100)
axes[1].barh(labels, values, color="#18678f")                    # length
axes[1].set_title("length"); axes[1].invert_yaxis()
axes[2].pie(values, labels=labels)                               # angle
axes[2].set_title("angle")
axes[3].scatter(range(5), [1] * 5, s=values * 40, color="#18678f")   # area
axes[3].set_title("area"); axes[3].set_yticks([])
for i, lab in enumerate(labels):
    axes[3].annotate(lab, (i, 1.005), ha="center")
axes[4].imshow([values], cmap="Blues", aspect="auto")            # colour value
axes[4].set_title("colour value"); axes[4].set_xticks(range(5))
axes[4].set_xticklabels(labels); axes[4].set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# TODO: your estimates of the ratio B/D, one per encoding, purely by eye:
my_estimates = {
    "position": None,   # e.g. 3.0
    "length": None,
    "angle": None,
    "area": None,
    "colour value": None,
}

# Verification — scores your perceptual error per channel.
true_ratio = 87 / 29
assert all(v is not None for v in my_estimates.values()), "Estimate every panel first"
errors = {k: round(abs(v - true_ratio) / true_ratio * 100) for k, v in my_estimates.items()}
for channel, err in sorted(errors.items(), key=lambda kv: kv[1]):
    print(f"{channel:>13}: {err:3d}% off")
print(f"\nTrue ratio B/D = {true_ratio:.2f}. "
      "Does your error ordering match the Cleveland & McGill ranking?")

## Exercise 2 — Wide to tidy, with the five checks

Analysts receive spreadsheets shaped for humans: one row per region, one
column per year. Charts (and Tableau, and pandas plotting, and AI-generated
code) want **tidy** data: each variable a column, each observation a row.

**Task.** The cell below builds the wide table. Reshape it to tidy long form —
ask your assistant, `pd.melt` is the move — then make the five verification
checks pass. Every assertion is one of the five checks from the lecture.

In [ ]:
# Build the wide table: annual riders per region (this cell is given).
annual = transport.assign(year=transport["date"].dt.year)
wide = (annual.groupby(["region", "year"])["riders"].sum()
        .unstack("year").reset_index())
wide.columns.name = None
wide

In [ ]:
def make_tidy(wide: pd.DataFrame) -> pd.DataFrame:
    """Reshape the wide region x year table to tidy long form.

    Returns a DataFrame with exactly three columns: region (str),
    year (int), riders (int) — one row per region-year observation.
    """
    # TODO: pd.melt, then fix dtypes
    raise NotImplementedError


tidy = make_tidy(wide)
tidy.head()

In [ ]:
# Verification — the five wrangling checks, as assertions.
# 1 IDENTIFIERS — no regions invented or dropped.
assert set(tidy["region"]) == set(REGIONS), "Region identifiers changed in the reshape"
# 2 ROW COUNT — 6 regions x 7 years = 42 observations, exactly.
assert len(tidy) == 42, f"Expected 42 rows (6 regions x 7 years), got {len(tidy)}"
# 3 TYPES — year and riders must be integers, not strings.
assert tidy["year"].dtype.kind == "i", "year should be an integer column"
assert tidy["riders"].dtype.kind == "i", "riders should be an integer column"
# 4 MISSING — the reshape must not create NaNs.
assert tidy.notna().all().all(), "NaNs appeared during the reshape"
# 5 SPOT TOTAL — one hand-checkable value survives the round trip.
spot = tidy.query("region == 'Inner Sydney' and year == 2019")["riders"].iloc[0]
expected = transport[(transport["region"] == "Inner Sydney")
                     & (transport["date"] < pd.Timestamp("2020-01-01"))]["riders"].sum()
assert spot == expected, "Inner Sydney 2019 total does not survive the reshape"
print("IDENTIFIERS ✓ ROWS ✓ TYPES ✓ MISSING ✓ SPOT TOTAL ✓ — tidy and trustworthy")

## Exercise 3 — The Visual Vocabulary sort

Classify each analytical question into **one** FT Visual Vocabulary category:

`deviation` · `correlation` · `ranking` · `distribution` · `change over time`
· `magnitude` · `part-to-whole` · `spatial` · `flow`

Then pick one question and draft the chart for it from `tidy` or `transport`.

In [ ]:
sort = {
    "Which regions have the highest ridership?": "...",
    "How has ridership changed since 2019?": "...",
    "Do train and bus ridership move together across months?": "...",
    "What share of 2025 riders does each mode contribute?": "...",
    "Where on the network are riders concentrated?": "...",
    "How do commuters move between home regions and work regions?": "...",
}
# TODO: replace each "..." with one category (exact strings listed above).

In [ ]:
# Verification — checks your sort against the answer key (some questions
# genuinely admit one neighbouring alternative; those are accepted).
key = {
    "Which regions have the highest ridership?": {"ranking", "magnitude"},
    "How has ridership changed since 2019?": {"change over time"},
    "Do train and bus ridership move together across months?": {"correlation"},
    "What share of 2025 riders does each mode contribute?": {"part-to-whole"},
    "Where on the network are riders concentrated?": {"spatial"},
    "How do commuters move between home regions and work regions?": {"flow"},
}
wrong = {q for q, cat in sort.items() if cat not in key[q]}
assert not wrong, "Reconsider these questions:\n- " + "\n- ".join(sorted(wrong))
print("Sort accepted. Now draft one of these charts below.")

In [ ]:
# Draft one chart for a question of your choice from the sort above.
# Write the docstring stating the task and the category, then let your
# assistant draft the body — then apply the verification ladder yourself.
def chart_for_question() -> None:
    """<state the question>

    FT Visual Vocabulary category: <state the category>
    Form chosen and why: <one sentence>
    """
    # TODO
    raise NotImplementedError


chart_for_question()

## Exercise 4 — Aspect ratio and baseline: same data, three impressions

The same series drawn in a wide frame, a tall frame, and as bars decides what
the reader feels before they think. None of these are "lies" — but each frame
privileges a different reading, and one of them is conventional for a reason.

**Task.** Implement one plotting function that honours the `figsize` it is
given, then a bar chart that must start at zero. Compare the impressions.

In [ ]:
inner = (transport[transport["region"] == "Inner Sydney"]
         .groupby("date")["riders"].sum())


def plot_series(series: pd.Series, figsize: tuple):
    """Line chart of the series in a figure of exactly `figsize` inches.
    Label the y-axis in millions. RETURN the Axes."""
    # TODO
    raise NotImplementedError


def plot_annual_bars(tidy: pd.DataFrame):
    """Bar chart of Inner Sydney annual riders from `tidy`.
    Bars encode length, so the y-axis MUST start at zero. RETURN the Axes."""
    # TODO
    raise NotImplementedError


ax_wide = plot_series(inner, (12, 2))
ax_tall = plot_series(inner, (4, 6))
ax_bars = plot_annual_bars(tidy)

In [ ]:
# Verification — the frames must be what they claim, the bars must be honest.
assert tuple(ax_wide.get_figure().get_size_inches()) == (12, 2), (
    "plot_series must honour the figsize it is given (wide)")
assert tuple(ax_tall.get_figure().get_size_inches()) == (4, 6), (
    "plot_series must honour the figsize it is given (tall)")
assert ax_bars.get_ylim()[0] == 0, "Bars encode length: the axis must start at zero"
print("Same data, three impressions. Which frame would each stakeholder choose?")

**Reflect** (edit this cell): the wide frame flattens the COVID collapse; the
tall frame makes it a cliff. Banking to ~45° is the conventional compromise —
which claim does each frame quietly make?

## Exercise 5 — Colour is an encoding

Three datasets, three palette decisions. Classify each scenario into the
palette family it needs — `sequential`, `diverging`, or `categorical` — then
prove one of them by drawing it.

In [ ]:
palette_choice = {
    "monthly ridership totals, low to high": "...",
    "percentage change in ridership vs the 2019 baseline (loss or gain)": "...",
    "the four transport modes on one chart": "...",
}
# TODO: replace each "..." with "sequential", "diverging", or "categorical".

In [ ]:
# Verification — palette families.
palette_key = {
    "monthly ridership totals, low to high": "sequential",
    "percentage change in ridership vs the 2019 baseline (loss or gain)": "diverging",
    "the four transport modes on one chart": "categorical",
}
wrong = [k for k, v in palette_choice.items() if v != palette_key[k]]
assert not wrong, "Reconsider: " + "; ".join(wrong)
print("Now prove the diverging one below.")

In [ ]:
def plot_change_heatmap(tidy: pd.DataFrame):
    """Heatmap of percentage change vs 2019, regions x years, with a
    DIVERGING colormap centred on zero (e.g. RdBu_r, vmin=-max, vmax=+max).
    RETURN the Axes."""
    # TODO: pivot tidy, compute pct change vs each region's 2019 value
    raise NotImplementedError


ax_h = plot_change_heatmap(tidy)

In [ ]:
# Verification — the colormap must actually be diverging and centred.
img = ax_h.get_images()[0] if ax_h.get_images() else None
assert img is not None, "Use imshow/pcolor-style heatmap so the colormap is inspectable"
assert img.get_cmap().name in {"RdBu", "RdBu_r", "coolwarm", "bwr", "seismic",
                               "PiYG", "PRGn", "BrBG", "RdYlBu", "RdYlBu_r"}, (
    f"'{img.get_cmap().name}' is not a diverging colormap")
lo, hi = img.get_clim()
assert abs(lo + hi) < max(abs(lo), abs(hi)) * 0.2, (
    "Centre the scale on zero: vmin and vmax should be symmetric")
print("Diverging, centred, honest. Loss and gain now read as different directions.")

## Exercise 6 — Redesign ×3

Here is a deliberately weak chart: an exploded, shadowed, rainbow pie of six
nearly-ordered values, titled with an exclamation mark instead of a claim.

**Task.** Diagnose what task the reader actually has, then produce **three
redesigns, each serving a different Visual Vocabulary category** (for example:
`ranking`, `change over time`, `part-to-whole` done honestly). This is the
studio artefact for today — you will present one of the three.

In [ ]:
# The weak chart — run, then diagnose. (Deliberately bad. Do not fix this cell.)
latest = (transport[transport["date"] == pd.Timestamp("2025-12-01")]
          .groupby("region")["riders"].sum())
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(latest, labels=latest.index, autopct="%1.1f%%", startangle=90,
       colors=plt.cm.rainbow(np.linspace(0, 1, len(latest))),
       explode=[0.1] * len(latest), shadow=True)
ax.set_title("Transport!!!")
plt.show()

In [ ]:
def redesign_ranking() -> None:
    """Redesign 1 — category: ranking.

    The reader's task: which regions have the most riders, in order?
    Form: sorted horizontal bar chart from a zero baseline, December 2025.
    Title states the finding, not the axes.
    """
    # TODO
    raise NotImplementedError


redesign_ranking()

In [ ]:
def redesign_change_over_time() -> None:
    """Redesign 2 — category: change over time.

    The reader's task: how did each region's ridership move through COVID
    and recovery? Form: line chart or small multiples from `transport`.
    """
    # TODO
    raise NotImplementedError


redesign_change_over_time()

In [ ]:
def redesign_third(category: str) -> None:
    """Redesign 3 — category: YOUR CHOICE (not ranking, not change over time).

    State the reader's task in one sentence here, choose an honest form,
    and pass the category you chose as `category`.
    """
    # TODO
    raise NotImplementedError


chosen_category = "..."  # TODO: e.g. "part-to-whole", "deviation", "distribution"
redesign_third(chosen_category)

In [ ]:
# Verification — three distinct categories, none of them the broken original's crime.
assert chosen_category not in {"...", "ranking", "change over time"}, (
    "Redesign 3 must use a category different from redesigns 1 and 2")
allowed = {"deviation", "correlation", "distribution", "magnitude",
           "part-to-whole", "spatial", "flow"}
assert chosen_category in allowed, f"'{chosen_category}' is not a Vocabulary category"
print("Three redesigns, three categories. For each, record: why this fits the "
      "task, what could mislead, and the simpler alternative you considered.")

**Studio hand-off.** Pick your strongest redesign. In the studio you will
present it with the four-line justification:

```text
FT Visual Vocabulary category:
Why this fits the task:
What could mislead:
Simpler alternative considered:
```

## AI disclosure block

Before you close this notebook, complete the course disclosure block. This is
the same block you will attach to every assessed artefact.

```text
What did the intelligent tool contribute?
  (e.g. "generated the first draft of plot_mode_recovery"; "suggested the melt call")
How was each contribution checked?
  (e.g. "ran the verification cell"; "hand-computed the Inner Sydney 2024 total")
What did you write or decide yourself?
What would you not trust the tool to do in this notebook?
```

Write your answers in the cell below.

*Your disclosure:*

- **Tool contributed:** …
- **How checked:** …
- **I wrote/decided:** …
- **Would not trust:** …